# Fake News Detection on LIAR with BERT (6-class)
Fine-tunes `bert-base-uncased` to classify political statements into six truthfulness levels.

In [ ]:
!pip install -q transformers datasets scikit-learn matplotlib torch

## 1. Imports and reproducibility

In [ ]:
import inspect
import numpy as np
import torch
import matplotlib.pyplot as plt
import transformers
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

SEED = 42
set_seed(SEED)
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

## 2. Load dataset

In [ ]:
# Load LIAR using its Parquet-based public mirror
dataset = load_dataset("tasksource/liar", split="train")

# Use 80% of the training split, as in the original notebook
dataset = dataset.select(range(int(len(dataset) * 0.8)))

print("Number of samples:", len(dataset))
print("Columns:", dataset.column_names)
print("First sample:", dataset[0])

## 3. Label names
Read the label names from the dataset itself instead of typing them by hand,
so the class order can never silently disagree with the data.

In [ ]:
label_feature = dataset.features["label"]
if hasattr(label_feature, "names"):
    label_names = label_feature.names
else:  # fallback to the original hand-written mapping
    label_names = ["false", "half-true", "mostly-true", "true", "barely-true", "pants-fire"]

id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in id2label.items()}
NUM_LABELS = len(label_names)
print("Labels:", id2label)

# Class distribution (LIAR is imbalanced, which is why we track macro-F1 too)
counts = np.bincount(dataset["label"], minlength=NUM_LABELS)
for i, c in enumerate(counts):
    print(f"{id2label[i]:>12}: {c} ({c / counts.sum():.1%})")

## 4. Train / evaluation split (seeded, stratified when possible)

In [ ]:
try:
    split_dataset = dataset.train_test_split(
        test_size=0.2, seed=SEED, stratify_by_column="label"
    )
except Exception:
    split_dataset = dataset.train_test_split(test_size=0.2, seed=SEED)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training samples:", len(train_dataset))
print("Evaluation samples:", len(eval_dataset))

## 5. Tokenization
Padding is now done **dynamically per batch** by the data collator instead of
padding every sample to 128 tokens, which is faster and uses less memory.

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["statement"], truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Keep only what the model needs
keep = {"input_ids", "attention_mask", "token_type_ids", "label"}
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep])
eval_dataset = eval_dataset.remove_columns([c for c in eval_dataset.column_names if c not in keep])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_dataset)

## 6. Model (6-class)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
print("Output classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

## 7. Training setup
Adds per-epoch evaluation, a learning rate and weight decay, best-checkpoint
selection by macro-F1, mixed precision on GPU, and disables W&B prompts.

In [ ]:
# Argument was renamed across transformers versions
eval_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="./fake-news-model",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=SEED,
    report_to="none",
    **{eval_arg: "epoch"},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer ready!")

## 8. Train

In [ ]:
trainer.train()

## 9. Evaluate

In [ ]:
results = trainer.evaluate()
print("Evaluation results:")
for key, value in results.items():
    print(f"{key}: {value}")

## 10. Detailed report and confusion matrix

In [ ]:
pred_output = trainer.predict(eval_dataset)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)
labels = list(range(NUM_LABELS))
names = [id2label[i] for i in labels]

print("CLASSIFICATION REPORT")
print(classification_report(y_true, y_pred, labels=labels, target_names=names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
print("CONFUSION MATRIX")
print(cm)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(labels); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(labels); ax.set_yticklabels(names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
for i in labels:
    for j in labels:
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
plt.tight_layout()
plt.show()

## 11. Predict on new statements
Returns the predicted truthfulness label with a confidence score and the top 3 candidates.

In [ ]:
def predict_news(text, top_k=3):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    top = torch.topk(probs, k=min(top_k, NUM_LABELS))
    ranked = [(model.config.id2label[i.item()], round(p.item(), 3)) for p, i in zip(top.values, top.indices)]
    return {"label": ranked[0][0], "confidence": ranked[0][1], "top_k": ranked}

claims = [
    "Says the unemployment rate has been cut in half since he took office.",
    "The state budget includes no new taxes for working families.",
    "Scientists have discovered a new planet.",
]
for c in claims:
    print(c)
    print("  ->", predict_news(c))

## 12. Save the model

In [ ]:
trainer.save_model("./fake-news-model-final")
tokenizer.save_pretrained("./fake-news-model-final")
print("Model and tokenizer saved successfully!")

## 13. (Optional) Download from Colab

In [ ]:
import shutil, os

archive = shutil.make_archive("fake-news-model-final", "zip", "fake-news-model-final")
print("Created:", archive)

try:  # only works inside Google Colab
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab; the zip is saved at", os.path.abspath(archive))